# Comparativa de modelos lineales con y sin ronda

Este notebook:
- se conecta a la base de datos,
- construye las features históricas,
- entrena 4 variantes del modelo lineal,
- evalúa cada una en el set histórico,
- y genera 4 tablas de partidos futuros.

Las 4 variantes son:
1. Lineal sin `round_number`
2. Lineal sin `round_number` + calibración
3. Lineal con `round_number`
4. Lineal con `round_number` + calibración

In [84]:
import sqlite3
import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score
from xgboost import XGBClassifier

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 200)

In [2]:
# Configuración general
DB_PATH = "../data/soccer.db"
ROLLING_WINDOW = 5
INITIAL_ELO = 1500
ELO_K = 20

TRAIN_MAX_SEASON = 2023
TEST_MIN_SEASON = 2024
THRESHOLD = 0.75

# Cambia aquí la liga que quieres probar en futuros partidos
FUTURE_COMPETITION = "Serie A"   # "Premier League", "Serie A", "Bundesliga", "Ligue 1", "La Liga"
FUTURE_SEASON = None             # None para no filtrar por temporada. O usa 2025 / 2026, etc.

In [3]:
conn = sqlite3.connect(DB_PATH)

query = '''
SELECT
    m.*,
    c.name AS competition_name
FROM matches m
JOIN competitions c
ON m.competition_id = c.id
'''

matches_df = pd.read_sql(query, conn)
matches_df["date"] = pd.to_datetime(matches_df["date"])
matches_df.head()

,id,api_fixture_id,competition_id,season,date,round,home_team_id,away_team_id,home_goals,away_goals,status,referee,competition_name
0,1,592143,1,2020,2020-09-12 11:30:00,Regular Season - 1,3,7,0.0,3.0,FT,C. Kavanagh,Premier League
1,2,592142,1,2020,2020-09-12 14:00:00,Regular Season - 1,16,6,1.0,0.0,FT,J. Moss,Premier League
2,3,592144,1,2020,2020-09-12 16:30:00,Regular Season - 1,5,19,4.0,3.0,FT,M. Oliver,Premier League
3,4,592148,1,2020,2020-09-12 19:00:00,Regular Season - 1,12,2,0.0,2.0,FT,S. Attwell,Premier League
4,5,592147,1,2020,2020-09-13 13:00:00,Regular Season - 1,17,10,0.0,3.0,FT,A. Taylor,Premier League


In [4]:
# Labels base
matches_df["result"] = matches_df.apply(
    lambda row: "H" if row["home_goals"] > row["away_goals"]
    else "A" if row["home_goals"] < row["away_goals"]
    else "D",
    axis=1
)

matches_df["total_goals"] = matches_df["home_goals"] + matches_df["away_goals"]
matches_df["over_2_5"] = matches_df["total_goals"] > 2.5
matches_df["btts"] = (matches_df["home_goals"] > 0) & (matches_df["away_goals"] > 0)
matches_df["is_draw"] = matches_df["home_goals"] == matches_df["away_goals"]
matches_df["home_win"] = matches_df["result"] == "H"
matches_df["away_win"] = matches_df["result"] == "A"
matches_df["home_dnb"] = matches_df["result"] != "A"
matches_df["away_dnb"] = matches_df["result"] != "H"

completed_matches = matches_df[
    matches_df["status"].isin(["FT", "AET", "PEN"])
].copy()

completed_matches = completed_matches.sort_values(by=["date", "id"]).copy()
completed_matches.head()

,id,api_fixture_id,competition_id,season,date,round,home_team_id,away_team_id,home_goals,away_goals,status,referee,competition_name,result,total_goals,over_2_5,btts,is_draw,home_win,away_win,home_dnb,away_dnb
18015,18016,143760,6,2016,2016-07-16 02:00:00,Apertura - 1,138,178,2.0,0.0,FT,NaN,Liga MX,H,2.0,False,False,False,True,False,True,False
18016,18017,143761,6,2016,2016-07-16 22:00:00,Apertura - 1,147,180,2.0,0.0,FT,NaN,Liga MX,H,2.0,False,False,False,True,False,True,False
18017,18018,143762,6,2016,2016-07-17 00:00:00,Apertura - 1,140,148,1.0,1.0,FT,NaN,Liga MX,D,2.0,False,True,True,False,False,True,True
18018,18019,143763,6,2016,2016-07-17 00:00:00,Apertura - 1,141,139,1.0,1.0,FT,NaN,Liga MX,D,2.0,False,True,True,False,False,True,True
18019,18020,143764,6,2016,2016-07-17 00:06:00,Apertura - 1,149,146,5.0,1.0,FT,NaN,Liga MX,H,6.0,True,True,False,True,False,True,False


In [5]:
# Historial de equipos para rolling features
home_df = completed_matches[
    [
        "date",
        "competition_name",
        "season",
        "home_team_id",
        "home_goals",
        "away_goals"
    ]
].copy()

home_df.columns = [
    "date",
    "competition_name",
    "season",
    "team_id",
    "goals_scored",
    "goals_conceded"
]

away_df = completed_matches[
    [
        "date",
        "competition_name",
        "season",
        "away_team_id",
        "away_goals",
        "home_goals"
    ]
].copy()

away_df.columns = [
    "date",
    "competition_name",
    "season",
    "team_id",
    "goals_scored",
    "goals_conceded"
]

team_history = pd.concat([home_df, away_df], ignore_index=True)
team_history = team_history.sort_values(by=["team_id", "date"]).copy()

team_history["rolling_goals_scored"] = (
    team_history.groupby("team_id")["goals_scored"]
    .transform(lambda x: x.shift(1).rolling(ROLLING_WINDOW).mean())
)

team_history["rolling_goals_conceded"] = (
    team_history.groupby("team_id")["goals_conceded"]
    .transform(lambda x: x.shift(1).rolling(ROLLING_WINDOW).mean())
)

team_history.head()

,date,competition_name,season,team_id,goals_scored,goals_conceded,rolling_goals_scored,rolling_goals_conceded
21305,2016-08-14 12:30:00,Premier League,2016,1,3.0,1.0,NaN,NaN
67,2016-08-19 19:00:00,Premier League,2016,1,2.0,0.0,NaN,NaN
21384,2016-08-27 16:30:00,Premier League,2016,1,1.0,0.0,NaN,NaN
177,2016-09-10 11:30:00,Premier League,2016,1,1.0,2.0,NaN,NaN
21516,2016-09-18 11:00:00,Premier League,2016,1,1.0,3.0,NaN,NaN


In [6]:
# Snapshot final por equipo para usar en partidos futuros
team_snapshot = (
    team_history.sort_values(["team_id", "date"])
    .groupby("team_id", as_index=False)
    .tail(1)
    [["team_id", "rolling_goals_scored", "rolling_goals_conceded"]]
    .copy()
)

# ELO histórico por partido + rating final por equipo
elo_matches = completed_matches.sort_values(by=["date", "id"]).copy()

elo_ratings = {}
elo_home = []
elo_away = []

for _, match in elo_matches.iterrows():
    home_team = match["home_team_id"]
    away_team = match["away_team_id"]

    home_elo = elo_ratings.get(home_team, INITIAL_ELO)
    away_elo = elo_ratings.get(away_team, INITIAL_ELO)

    elo_home.append(home_elo)
    elo_away.append(away_elo)

    expected_home = 1 / (1 + 10 ** ((away_elo - home_elo) / 400))
    expected_away = 1 - expected_home

    if match["home_goals"] > match["away_goals"]:
        actual_home, actual_away = 1, 0
    elif match["home_goals"] < match["away_goals"]:
        actual_home, actual_away = 0, 1
    else:
        actual_home, actual_away = 0.5, 0.5

    elo_ratings[home_team] = home_elo + ELO_K * (actual_home - expected_home)
    elo_ratings[away_team] = away_elo + ELO_K * (actual_away - expected_away)

elo_matches["home_elo"] = elo_home
elo_matches["away_elo"] = elo_away

final_elo_df = pd.DataFrame(
    [{"team_id": team_id, "elo": elo} for team_id, elo in elo_ratings.items()]
)

team_snapshot = team_snapshot.merge(final_elo_df, on="team_id", how="left")
team_snapshot.head()

,team_id,rolling_goals_scored,rolling_goals_conceded,elo
0,1,1.4,1.0,1648.436071
1,2,1.2,1.4,1585.997824
2,3,0.2,1.2,1551.826958
3,4,0.2,2.4,1434.516578
4,5,2.0,1.2,1662.775859


In [7]:
# Features históricas para entrenamiento
home_features = team_history.rename(
    columns={
        "team_id": "home_team_id",
        "rolling_goals_scored": "home_rolling_scored",
        "rolling_goals_conceded": "home_rolling_conceded",
    }
)

away_features = team_history.rename(
    columns={
        "team_id": "away_team_id",
        "rolling_goals_scored": "away_rolling_scored",
        "rolling_goals_conceded": "away_rolling_conceded",
    }
)

matches_features = completed_matches.merge(
    home_features[
        [
            "date",
            "home_team_id",
            "home_rolling_scored",
            "home_rolling_conceded"
        ]
    ],
    on=["date", "home_team_id"],
    how="left"
)

matches_features = matches_features.merge(
    away_features[
        [
            "date",
            "away_team_id",
            "away_rolling_scored",
            "away_rolling_conceded"
        ]
    ],
    on=["date", "away_team_id"],
    how="left"
)

matches_features = matches_features.merge(
    elo_matches[["id", "home_elo", "away_elo"]],
    on="id",
    how="left"
)

matches_features["round_number"] = (
    matches_features["round"]
    .astype(str)
    .str.extract(r"(\d+)")[0]
    .astype(float)
)

matches_features.head()

,id,api_fixture_id,competition_id,season,date,round,home_team_id,away_team_id,home_goals,away_goals,status,referee,competition_name,result,total_goals,over_2_5,btts,is_draw,home_win,away_win,home_dnb,away_dnb,home_rolling_scored,home_rolling_conceded,away_rolling_scored,away_rolling_conceded,home_elo,away_elo,round_number
0,18016,143760,6,2016,2016-07-16 02:00:00,Apertura - 1,138,178,2.0,0.0,FT,NaN,Liga MX,H,2.0,False,False,False,True,False,True,False,NaN,NaN,NaN,NaN,1500.0,1500.0,1.0
1,18017,143761,6,2016,2016-07-16 22:00:00,Apertura - 1,147,180,2.0,0.0,FT,NaN,Liga MX,H,2.0,False,False,False,True,False,True,False,NaN,NaN,NaN,NaN,1500.0,1500.0,1.0
2,18018,143762,6,2016,2016-07-17 00:00:00,Apertura - 1,140,148,1.0,1.0,FT,NaN,Liga MX,D,2.0,False,True,True,False,False,True,True,NaN,NaN,NaN,NaN,1500.0,1500.0,1.0
3,18019,143763,6,2016,2016-07-17 00:00:00,Apertura - 1,141,139,1.0,1.0,FT,NaN,Liga MX,D,2.0,False,True,True,False,False,True,True,NaN,NaN,NaN,NaN,1500.0,1500.0,1.0
4,18020,143764,6,2016,2016-07-17 00:06:00,Apertura - 1,149,146,5.0,1.0,FT,NaN,Liga MX,H,6.0,True,True,False,True,False,True,False,NaN,NaN,NaN,NaN,1500.0,1500.0,1.0


In [8]:
def build_future_matches(matches_df: pd.DataFrame,
                         team_snapshot: pd.DataFrame,
                         competition_name: str,
                         season: int | None = None) -> pd.DataFrame:
    future_mask = (
        (matches_df["status"] == "NS") &
        (matches_df["competition_name"] == competition_name)
    )
    if season is not None:
        future_mask &= (matches_df["season"] == season)

    future_matches = matches_df.loc[future_mask].copy()
    future_matches["round_number"] = (
        future_matches["round"]
        .astype(str)
        .str.extract(r"(\d+)")[0]
        .astype(float)
    )

    teams_df = pd.read_sql(
        "SELECT id, name FROM teams",
        conn
    )
    team_map = dict(zip(teams_df["id"], teams_df["name"]))

    future_matches["home_team_name"] = future_matches["home_team_id"].map(team_map)
    future_matches["away_team_name"] = future_matches["away_team_id"].map(team_map)

    home_snapshot = team_snapshot.rename(
        columns={
            "team_id": "home_team_id",
            "rolling_goals_scored": "home_rolling_scored",
            "rolling_goals_conceded": "home_rolling_conceded",
            "elo": "home_elo",
        }
    )

    away_snapshot = team_snapshot.rename(
        columns={
            "team_id": "away_team_id",
            "rolling_goals_scored": "away_rolling_scored",
            "rolling_goals_conceded": "away_rolling_conceded",
            "elo": "away_elo",
        }
    )

    future_matches = future_matches.merge(
        home_snapshot[
            [
                "home_team_id",
                "home_rolling_scored",
                "home_rolling_conceded",
                "home_elo"
            ]
        ],
        on="home_team_id",
        how="left"
    )

    future_matches = future_matches.merge(
        away_snapshot[
            [
                "away_team_id",
                "away_rolling_scored",
                "away_rolling_conceded",
                "away_elo"
            ]
        ],
        on="away_team_id",
        how="left"
    )

    return future_matches


def fit_and_score_linear(train_df: pd.DataFrame,
                         test_df: pd.DataFrame,
                         feature_cols: list[str],
                         target: str = "home_dnb",
                         calibrated: bool = False,
                         threshold: float = 0.75):
    model_df = train_df.dropna(subset=feature_cols + [target]).copy()
    future_df = test_df.dropna(subset=feature_cols).copy()

    train_split = model_df[model_df["season"] <= TRAIN_MAX_SEASON].copy()
    test_split = model_df[model_df["season"] >= TEST_MIN_SEASON].copy()

    X_train = train_split[feature_cols]
    y_train = train_split[target]

    X_test = test_split[feature_cols]
    y_test = test_split[target]

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    base_model = LogisticRegression(max_iter=1000)

    if calibrated:
        model = CalibratedClassifierCV(
            base_model,
            method="sigmoid",
            cv=5
        )
    else:
        model = base_model

    model.fit(X_train_scaled, y_train)

    test_predictions = model.predict(X_test_scaled)
    test_probabilities = model.predict_proba(X_test_scaled)[:, 1]

    test_results = test_split.copy()
    test_results["prediction_proba"] = test_probabilities
    test_results["prediction"] = test_results["prediction_proba"] >= threshold

    confident_predictions = test_results[
        test_results["prediction_proba"] >= threshold
    ].copy()

    away_confident_predictions = test_results[
        test_results["prediction_proba"] <= 0.25
    ].copy()

    home_band_predictions = test_results[
        (
            test_results["prediction_proba"] >= 0.59
        ) &
        (
            test_results["prediction_proba"] <= 0.74
        )
    ].copy()

    draw_band_predictions = test_results[
        (
            test_results["prediction_proba"] >= 0.25
        ) &
        (
            test_results["prediction_proba"] <= 0.40
        )
    ].copy()

    away_band_predictions = test_results[
        (
            test_results["prediction_proba"] >= 0.26
        ) &
        (
            test_results["prediction_proba"] <= 0.59
        )
    ].copy()

    metrics = {
        "test_accuracy": float(
            accuracy_score(
                y_test,
                test_predictions
            )
        ),
        "high_confidence_accuracy": float(
            accuracy_score(
                confident_predictions["home_win"],
                confident_predictions["prediction"]
            )
        ) if len(confident_predictions) else np.nan,
        "high_confidence_count": int(
            len(confident_predictions)
        ),
        "high_away_accuracy": float(
            away_confident_predictions["away_win"].mean()
        ) if len(away_confident_predictions) else np.nan,
        "high_away_count": int(
            len(away_confident_predictions)
        ),
        "home_band_41_74_accuracy": float(
            home_band_predictions["home_dnb"].mean()
        ) if len(home_band_predictions) else np.nan,
        "home_band_41_74_count": int(
            len(home_band_predictions)
        ),
        "draw_band_25_40_accuracy": float(
            draw_band_predictions["is_draw"].mean()
        ) if len(draw_band_predictions) else np.nan,
        "draw_band_25_40_count": int(
            len(draw_band_predictions)
        ),
        "away_band_26_59_accuracy": float(
            away_band_predictions["away_dnb"].mean()
        ) if len(away_band_predictions) else np.nan,
        "away_band_26_59_count": int(
            len(away_band_predictions)
        )
    }

    future_X = future_df[feature_cols]
    future_X_scaled = scaler.transform(future_X)
    future_probabilities = model.predict_proba(future_X_scaled)[:, 1]

    future_results = future_df.copy()
    future_results["prediction_proba"] = future_probabilities
    future_results["prediction"] = future_results["prediction_proba"] >= threshold
    future_results = future_results.sort_values(by="prediction_proba", ascending=False)

    return {
        "model": model,
        "scaler": scaler,
        "test_results": test_results,
        "future_results": future_results,
        "metrics": metrics,
    }

In [9]:
# Partidos futuros: cambia la liga aquí
future_matches = build_future_matches(
    matches_df=matches_df,
    team_snapshot=team_snapshot,
    competition_name=FUTURE_COMPETITION,
    season=FUTURE_SEASON
)

future_matches.head()

,id,api_fixture_id,competition_id,season,date,round,home_team_id,away_team_id,home_goals,away_goals,status,referee,competition_name,result,total_goals,over_2_5,btts,is_draw,home_win,away_win,home_dnb,away_dnb,round_number,home_team_name,away_team_name,home_rolling_scored,home_rolling_conceded,home_elo,away_rolling_scored,away_rolling_conceded,away_elo
0,20486,1378237,3,2025,2026-05-23 18:45:00,Regular Season - 38,55,183,NaN,NaN,NS,NaN,Serie A,D,NaN,False,False,False,False,False,True,True,38.0,Lazio,Pisa,1.4,1.6,1608.951737,0.4,2.2,1395.590424
1,20487,1378239,3,2025,2026-05-24 18:45:00,Regular Season - 38,57,58,NaN,NaN,NS,NaN,Serie A,D,NaN,False,False,False,False,False,True,True,38.0,AC Milan,Cagliari,0.6,1.6,1672.641323,0.8,1.4,1487.311115
2,20488,1378240,3,2025,2026-05-24 16:00:00,Regular Season - 38,59,60,NaN,NaN,NS,NaN,Serie A,D,NaN,False,False,False,False,False,True,True,38.0,Napoli,Udinese,1.4,1.2,1705.962825,2.0,0.8,1538.223537
3,20489,1378234,3,2025,2026-05-23 16:00:00,Regular Season - 38,66,71,NaN,NaN,NS,NaN,Serie A,D,NaN,False,False,False,False,False,True,True,38.0,Bologna,Inter,1.0,1.2,1601.680868,2.8,1.0,1796.953860
4,20490,1378236,3,2025,2026-05-22 18:45:00,Regular Season - 38,68,65,NaN,NaN,NS,NaN,Serie A,D,NaN,False,False,False,False,False,True,True,38.0,Fiorentina,Atalanta,0.4,1.0,1578.608210,1.2,1.4,1647.733036


In [10]:
# =========================
# MODELO 1: LINEAL SIN RONDA
# =========================
features_no_round = [
    "home_rolling_scored",
    "away_rolling_scored",
    "home_rolling_conceded",
    "away_rolling_conceded",
    "home_elo",
    "away_elo",
]

result_linear_no_round = fit_and_score_linear(
    train_df=matches_features,
    test_df=future_matches,
    feature_cols=features_no_round,
    target="home_dnb",
    calibrated=False,
    threshold=THRESHOLD
)

print(classification_report(
    result_linear_no_round["test_results"]["home_dnb"],
    result_linear_no_round["test_results"]["prediction"]
))

print(result_linear_no_round["metrics"])

future_table_linear_no_round = result_linear_no_round["future_results"][
    [
        "date",
        "competition_name",
        "home_team_name",
        "away_team_name",
        "prediction_proba"
    ]
].copy()

future_table_linear_no_round.head(50)

              precision    recall  f1-score   support

       False       0.42      0.77      0.54      1281
        True       0.83      0.51      0.63      2836

    accuracy                           0.59      4117
   macro avg       0.63      0.64      0.59      4117
weighted avg       0.70      0.59      0.60      4117

{'test_accuracy': 0.7099829973281516, 'high_confidence_accuracy': 0.6042626728110599, 'high_confidence_count': 1736, 'high_away_accuracy': 0.7291666666666666, 'high_away_count': 48, 'home_band_41_74_accuracy': 0.6785095320623917, 'home_band_41_74_count': 1154, 'draw_band_25_40_accuracy': 0.23137254901960785, 'draw_band_25_40_count': 255, 'away_band_26_59_accuracy': 0.7808090310442145, 'away_band_26_59_count': 1063}


,date,competition_name,home_team_name,away_team_name,prediction_proba
0,2026-05-23 18:45:00,Serie A,Lazio,Pisa,0.909746
1,2026-05-24 18:45:00,Serie A,AC Milan,Cagliari,0.876172
2,2026-05-24 16:00:00,Serie A,Napoli,Udinese,0.859139
8,2026-05-24 13:00:00,Serie A,Parma,Sassuolo,0.698140
9,2026-05-24 18:45:00,Serie A,Lecce,Genoa,0.651205
4,2026-05-22 18:45:00,Serie A,Fiorentina,Atalanta,0.606189
5,2026-05-24 18:45:00,Serie A,Torino,Juventus,0.487426
7,2026-05-24 18:45:00,Serie A,Cremonese,Como,0.428281
3,2026-05-23 16:00:00,Serie A,Bologna,Inter,0.381036
6,2026-05-24 18:45:00,Serie A,Hellas Verona,AS Roma,0.290465


In [11]:
# ===========================================
# MODELO 2: LINEAL SIN RONDA + CALIBRACIÓN
# ===========================================
result_linear_no_round_cal = fit_and_score_linear(
    train_df=matches_features,
    test_df=future_matches,
    feature_cols=features_no_round,
    target="home_dnb",
    calibrated=True,
    threshold=THRESHOLD
)

print(classification_report(
    result_linear_no_round_cal["test_results"]["home_dnb"],
    result_linear_no_round_cal["test_results"]["prediction"]
))

print(result_linear_no_round_cal["metrics"])

future_table_linear_no_round_cal = result_linear_no_round_cal["future_results"][
    [
        "date",
        "competition_name",
        "home_team_name",
        "away_team_name",
        "prediction_proba"
    ]
].copy()

future_table_linear_no_round_cal.head(50)

              precision    recall  f1-score   support

       False       0.42      0.77      0.54      1281
        True       0.83      0.51      0.63      2836

    accuracy                           0.59      4117
   macro avg       0.63      0.64      0.59      4117
weighted avg       0.70      0.59      0.61      4117

{'test_accuracy': 0.7121690551372358, 'high_confidence_accuracy': 0.6037844036697247, 'high_confidence_count': 1744, 'high_away_accuracy': 0.7358490566037735, 'high_away_count': 53, 'home_band_41_74_accuracy': 0.6811722912966253, 'home_band_41_74_count': 1126, 'draw_band_25_40_accuracy': 0.22916666666666666, 'draw_band_25_40_count': 288, 'away_band_26_59_accuracy': 0.7758139534883721, 'away_band_26_59_count': 1075}


,date,competition_name,home_team_name,away_team_name,prediction_proba
0,2026-05-23 18:45:00,Serie A,Lazio,Pisa,0.911974
1,2026-05-24 18:45:00,Serie A,AC Milan,Cagliari,0.879035
2,2026-05-24 16:00:00,Serie A,Napoli,Udinese,0.862244
8,2026-05-24 13:00:00,Serie A,Parma,Sassuolo,0.697868
9,2026-05-24 18:45:00,Serie A,Lecce,Genoa,0.649406
4,2026-05-22 18:45:00,Serie A,Fiorentina,Atalanta,0.601440
5,2026-05-24 18:45:00,Serie A,Torino,Juventus,0.478374
7,2026-05-24 18:45:00,Serie A,Cremonese,Como,0.418064
3,2026-05-23 16:00:00,Serie A,Bologna,Inter,0.370600
6,2026-05-24 18:45:00,Serie A,Hellas Verona,AS Roma,0.281755


In [12]:
# =========================
# MODELO 3: LINEAL CON RONDA
# =========================
features_with_round = features_no_round + ["round_number"]

result_linear_with_round = fit_and_score_linear(
    train_df=matches_features,
    test_df=future_matches,
    feature_cols=features_with_round,
    target="home_dnb",
    calibrated=False,
    threshold=THRESHOLD
)

print(classification_report(
    result_linear_with_round["test_results"]["home_dnb"],
    result_linear_with_round["test_results"]["prediction"]
))

print(result_linear_with_round["metrics"])

future_table_linear_with_round = result_linear_with_round["future_results"][
    [
        "date",
        "competition_name",
        "home_team_name",
        "away_team_name",
        "prediction_proba"
    ]
].copy()

future_table_linear_with_round.head(50)

              precision    recall  f1-score   support

       False       0.42      0.78      0.55      1270
        True       0.84      0.51      0.64      2782

    accuracy                           0.60      4052
   macro avg       0.63      0.65      0.59      4052
weighted avg       0.71      0.60      0.61      4052

{'test_accuracy': 0.707305034550839, 'high_confidence_accuracy': 0.6051401869158879, 'high_confidence_count': 1712, 'high_away_accuracy': 0.74, 'high_away_count': 50, 'home_band_41_74_accuracy': 0.6717352415026834, 'home_band_41_74_count': 1118, 'draw_band_25_40_accuracy': 0.23846153846153847, 'draw_band_25_40_count': 260, 'away_band_26_59_accuracy': 0.782650142993327, 'away_band_26_59_count': 1049}


,date,competition_name,home_team_name,away_team_name,prediction_proba
0,2026-05-23 18:45:00,Serie A,Lazio,Pisa,0.908991
1,2026-05-24 18:45:00,Serie A,AC Milan,Cagliari,0.874659
2,2026-05-24 16:00:00,Serie A,Napoli,Udinese,0.856276
8,2026-05-24 13:00:00,Serie A,Parma,Sassuolo,0.694022
9,2026-05-24 18:45:00,Serie A,Lecce,Genoa,0.646977
4,2026-05-22 18:45:00,Serie A,Fiorentina,Atalanta,0.602487
5,2026-05-24 18:45:00,Serie A,Torino,Juventus,0.480337
7,2026-05-24 18:45:00,Serie A,Cremonese,Como,0.423379
3,2026-05-23 16:00:00,Serie A,Bologna,Inter,0.374887
6,2026-05-24 18:45:00,Serie A,Hellas Verona,AS Roma,0.285321


In [13]:
# ===========================================
# MODELO 4: LINEAL CON RONDA + CALIBRACIÓN
# ===========================================
result_linear_with_round_cal = fit_and_score_linear(
    train_df=matches_features,
    test_df=future_matches,
    feature_cols=features_with_round,
    target="home_dnb",
    calibrated=True,
    threshold=THRESHOLD
)

print(classification_report(
    result_linear_with_round_cal["test_results"]["home_dnb"],
    result_linear_with_round_cal["test_results"]["prediction"]
))

print(result_linear_with_round_cal["metrics"])

future_table_linear_with_round_cal = result_linear_with_round_cal["future_results"][
    [
        "date",
        "competition_name",
        "home_team_name",
        "away_team_name",
        "prediction_proba"
    ]
].copy()

future_table_linear_with_round_cal.head(50)

              precision    recall  f1-score   support

       False       0.42      0.78      0.55      1270
        True       0.84      0.52      0.64      2782

    accuracy                           0.60      4052
   macro avg       0.63      0.65      0.59      4052
weighted avg       0.71      0.60      0.61      4052

{'test_accuracy': 0.7105133267522211, 'high_confidence_accuracy': 0.6046376811594203, 'high_confidence_count': 1725, 'high_away_accuracy': 0.7407407407407407, 'high_away_count': 54, 'home_band_41_74_accuracy': 0.6724770642201835, 'home_band_41_74_count': 1090, 'draw_band_25_40_accuracy': 0.2264808362369338, 'draw_band_25_40_count': 287, 'away_band_26_59_accuracy': 0.7763157894736842, 'away_band_26_59_count': 1064}


,date,competition_name,home_team_name,away_team_name,prediction_proba
0,2026-05-23 18:45:00,Serie A,Lazio,Pisa,0.910435
1,2026-05-24 18:45:00,Serie A,AC Milan,Cagliari,0.876533
2,2026-05-24 16:00:00,Serie A,Napoli,Udinese,0.858396
8,2026-05-24 13:00:00,Serie A,Parma,Sassuolo,0.693393
9,2026-05-24 18:45:00,Serie A,Lecce,Genoa,0.645143
4,2026-05-22 18:45:00,Serie A,Fiorentina,Atalanta,0.597977
5,2026-05-24 18:45:00,Serie A,Torino,Juventus,0.472254
7,2026-05-24 18:45:00,Serie A,Cremonese,Como,0.414708
3,2026-05-23 16:00:00,Serie A,Bologna,Inter,0.366166
6,2026-05-24 18:45:00,Serie A,Hellas Verona,AS Roma,0.278706


In [14]:
# Comparación de los 4 modelos
comparison_summary = pd.DataFrame([
    {
        "model": "Linear sin ronda",
        **result_linear_no_round["metrics"],
    },
    {
        "model": "Linear sin ronda + calibración",
        **result_linear_no_round_cal["metrics"],
    },
    {
        "model": "Linear con ronda",
        **result_linear_with_round["metrics"],
    },
    {
        "model": "Linear con ronda + calibración",
        **result_linear_with_round_cal["metrics"],
    },
])

comparison_summary = comparison_summary.sort_values(
    by=["high_confidence_accuracy", "test_accuracy"],
    ascending=False
)

comparison_summary

,model,test_accuracy,high_confidence_accuracy,high_confidence_count,high_away_accuracy,high_away_count,home_band_41_74_accuracy,home_band_41_74_count,draw_band_25_40_accuracy,draw_band_25_40_count,away_band_26_59_accuracy,away_band_26_59_count
2,Linear con ronda,0.707305,0.605140,1712,0.740000,50,0.671735,1118,0.238462,260,0.782650,1049
3,Linear con ronda + calibración,0.710513,0.604638,1725,0.740741,54,0.672477,1090,0.226481,287,0.776316,1064
0,Linear sin ronda,0.709983,0.604263,1736,0.729167,48,0.678510,1154,0.231373,255,0.780809,1063
1,Linear sin ronda + calibración,0.712169,0.603784,1744,0.735849,53,0.681172,1126,0.229167,288,0.775814,1075


### Intento de modelo de empates

## Notas rápidas

- Para probar otra liga, cambia `FUTURE_COMPETITION`.
- Si quieres acotar por temporada, cambia `FUTURE_SEASON`.
- Las 4 tablas futuras quedan en estas variables:
  - `future_table_linear_no_round`
  - `future_table_linear_no_round_cal`
  - `future_table_linear_with_round`
  - `future_table_linear_with_round_cal`

In [80]:
print(multiclass_df.columns.tolist())

['id', 'api_fixture_id', 'competition_id', 'season', 'date', 'round', 'home_team_id', 'away_team_id', 'home_goals', 'away_goals', 'status', 'referee', 'competition_name', 'result', 'total_goals', 'over_2_5', 'btts', 'is_draw', 'home_win', 'away_win', 'home_dnb', 'away_dnb', 'home_rolling_scored', 'home_rolling_conceded', 'away_rolling_scored', 'away_rolling_conceded', 'home_elo', 'away_elo', 'round_number', 'elo_diff', 'attack_diff', 'defense_diff', 'combined_scoring', 'combined_conceded', 'total_balance', 'match_result_3class', 'home_advantage_strength', 'away_advantage_strength', 'total_match_volatility', 'low_balance_flag']


In [81]:
multiclass_features = [

    # rolling
    "home_rolling_scored",
    "away_rolling_scored",

    "home_rolling_conceded",
    "away_rolling_conceded",

    # elo
    "home_elo",
    "away_elo",

    # parity
    "elo_diff",
    "attack_diff",
    "defense_diff",

    # pace
    "combined_scoring",
    "combined_conceded",

    # balance
    "total_balance",

    # playoffs
    "is_playoff"
]

In [85]:
# ===========================================
# MULTICLASS XGBOOST
# ===========================================

target_map = {
    "home": 0,
    "draw": 1,
    "away": 2
}

multiclass_df = matches_features.copy()

multiclass_df["elo_diff"] = (
    multiclass_df["home_elo"] -
    multiclass_df["away_elo"]
).abs()

multiclass_df["attack_diff"] = (
    multiclass_df["home_rolling_scored"] -
    multiclass_df["away_rolling_scored"]
).abs()

multiclass_df["defense_diff"] = (
    multiclass_df["home_rolling_conceded"] -
    multiclass_df["away_rolling_conceded"]
).abs()

multiclass_df["combined_scoring"] = (
    multiclass_df["home_rolling_scored"] +
    multiclass_df["away_rolling_scored"]
)

multiclass_df["combined_conceded"] = (
    multiclass_df["home_rolling_conceded"] +
    multiclass_df["away_rolling_conceded"]
)

multiclass_df["total_balance"] = (
    multiclass_df["attack_diff"] +
    multiclass_df["defense_diff"]
)

# Target multiclass
multiclass_df["match_result_3class"] = np.select(
    [
        multiclass_df["home_goals"] > multiclass_df["away_goals"],
        multiclass_df["home_goals"] == multiclass_df["away_goals"],
        multiclass_df["home_goals"] < multiclass_df["away_goals"]
    ],
    [0, 1, 2]
)

# Features base
multiclass_df["elo_diff"] = (
    multiclass_df["home_elo"] -
    multiclass_df["away_elo"]
).abs()

multiclass_df["attack_diff"] = (
    multiclass_df["home_rolling_scored"] -
    multiclass_df["away_rolling_scored"]
).abs()

multiclass_df["defense_diff"] = (
    multiclass_df["home_rolling_conceded"] -
    multiclass_df["away_rolling_conceded"]
).abs()

multiclass_df["combined_scoring"] = (
    multiclass_df["home_rolling_scored"] +
    multiclass_df["away_rolling_scored"]
)

multiclass_df["total_balance"] = (
    multiclass_df["attack_diff"] +
    multiclass_df["defense_diff"]
)

multiclass_df["home_advantage_strength"] = (
    multiclass_df["home_rolling_scored"] -
    multiclass_df["away_rolling_conceded"]
)

multiclass_df["away_advantage_strength"] = (
    multiclass_df["away_rolling_scored"] -
    multiclass_df["home_rolling_conceded"]
)

multiclass_df["total_match_volatility"] = (
    multiclass_df["combined_scoring"] +
    multiclass_df["home_rolling_conceded"] +
    multiclass_df["away_rolling_conceded"]
)

multiclass_df["low_balance_flag"] = (
    multiclass_df["total_balance"] <= 1
).astype(int)

# Si ya tienes una probabilidad del modelo base, úsala aquí
# multiclass_df["home_dnb_probability"] = ...
# multiclass_df["home_distance_from_50"] = (multiclass_df["home_dnb_probability"] - 0.5).abs()

multiclass_features = [
    "home_rolling_scored",
    "away_rolling_scored",

    "home_rolling_conceded",
    "away_rolling_conceded",

    "home_elo",
    "away_elo",

    "elo_diff",
    "attack_diff",
    "defense_diff",

    "combined_scoring",
    "total_balance",

    "home_advantage_strength",
    "away_advantage_strength",

    "total_match_volatility",

    "low_balance_flag",

    "is_playoff"
]
multiclass_df["round_str"] = (
    multiclass_df["round"]
    .astype(str)
)

multiclass_df["is_playoff"] = (
    multiclass_df["round_str"]
    .str.contains(
        r"Quarter|Semi|Final|Reclas|Liguilla|Play-In",
        case=False,
        na=False
    )
    .astype(int)
)
multiclass_df = multiclass_df.dropna(
    subset=multiclass_features + ["match_result_3class"]
).copy()

train_df = multiclass_df[multiclass_df["season"] <= TRAIN_MAX_SEASON].copy()
test_df = multiclass_df[multiclass_df["season"] >= TEST_MIN_SEASON].copy()

X_train = train_df[multiclass_features]
y_train = train_df["match_result_3class"]

X_test = test_df[multiclass_features]
y_test = test_df["match_result_3class"]

# Balance por clase
class_counts = y_train.value_counts().to_dict()
sample_weight = y_train.map(lambda c: max(class_counts.values()) / class_counts[c])

xgb_model = XGBClassifier(
    objective="multi:softprob",
    num_class=3,
    n_estimators=500,
    max_depth=4,
    learning_rate=0.03,
    subsample=0.85,
    colsample_bytree=0.85,
    min_child_weight=3,
    reg_lambda=1.0,
    random_state=42
)

xgb_model.fit(
    X_train,
    y_train,
    sample_weight=sample_weight,
    eval_set=[(X_test, y_test)],
    verbose=False
)

predictions = xgb_model.predict(X_test)
print(classification_report(y_test, predictions))

              precision    recall  f1-score   support

           0       0.60      0.58      0.59      1803
           1       0.28      0.23      0.25      1033
           2       0.48      0.57      0.52      1281

    accuracy                           0.49      4117
   macro avg       0.45      0.46      0.45      4117
weighted avg       0.48      0.49      0.48      4117



In [95]:
probabilities = xgb_model.predict_proba(X_test)

test_results = test_df.copy()

test_results["home_prob"] = probabilities[:, 0]
test_results["draw_prob"] = probabilities[:, 1]
test_results["away_prob"] = probabilities[:, 2]

teams_query = """
SELECT
    id,
    name
FROM teams
"""

teams_df = pd.read_sql(
    teams_query,
    conn
)

team_map = dict(
    zip(
        teams_df["id"],
        teams_df["name"]
    )
)

test_results["home_team_name"] = (
    test_results["home_team_id"]
    .map(team_map)
)

test_results["away_team_name"] = (
    test_results["away_team_id"]
    .map(team_map)
)

# IMPORTANTE:
# crear high_draws DESPUÉS
high_draws = test_results[
    test_results["draw_prob"] >= 0.33
].copy()

high_draws[
    [
        "competition_name",

        "season",

        "date",

        "home_team_name",

        "away_team_name",

        "home_prob",

        "draw_prob",

        "away_prob",

        "match_result_3class"
    ]
].sort_values(
    by="draw_prob",
    ascending=False
).head(50)

,competition_name,season,date,home_team_name,away_team_name,home_prob,draw_prob,away_prob,match_result_3class
19991,La Liga,2025,2025-12-02 20:00:00,Barcelona,Atletico Madrid,0.269804,0.637722,0.092474,0
19790,Bundesliga,2025,2025-11-01 17:30:00,Bayern München,Bayer Leverkusen,0.300197,0.590799,0.109004,0
17256,Premier League,2024,2024-08-31 14:00:00,Nottingham Forest,Wolves,0.247302,0.560333,0.192364,1
17408,Premier League,2024,2024-09-22 15:30:00,Manchester City,Arsenal,0.386037,0.521458,0.092504,1
17571,Serie A,2024,2024-10-19 18:45:00,Juventus,Lazio,0.238921,0.510112,0.250967,0
18540,Premier League,2024,2025-02-27 20:00:00,West Ham,Leicester,0.310542,0.509357,0.180101,0
20982,Liga MX,2025,2026-04-19 03:10:00,Club America,Toluca,0.237645,0.489059,0.273295,0
19602,La Liga,2025,2025-10-05 12:00:00,Alaves,Elche,0.263594,0.486801,0.249605,0
20182,Ligue 1,2025,2026-01-03 18:00:00,Nice,Strasbourg,0.252158,0.486216,0.261626,1
17241,Serie A,2024,2024-08-30 18:45:00,Inter,Atalanta,0.381186,0.486144,0.132669,0


In [102]:
test_results[
    [
        "draw_prob",
        "calibrated_draw_prob"
    ]
].sort_values(
    by="calibrated_draw_prob",
    ascending=False
).head(30)

,draw_prob,calibrated_draw_prob
19991,0.637722,0.309432
19790,0.590799,0.300440
17256,0.560333,0.294682
17408,0.521458,0.287429
17571,0.510112,0.285332
18540,0.509357,0.285193
20982,0.489059,0.281466
19602,0.486801,0.281053
20182,0.486216,0.280946
17241,0.486144,0.280933


### Probando mejorar el modelo

In [110]:
matches_query = """
SELECT
    *
FROM matches
"""

matches_df = pd.read_sql(
    matches_query,
    conn
)
context_query = """
SELECT
    match_id,

    home_position,
    away_position,

    points_diff,
    position_diff,

    home_points,
    away_points,

    home_rest_days,
    away_rest_days,

    home_title_race,
    away_title_race,

    away_europe_race,
    home_europe_race,
    
    home_relegation_risk,
    away_relegation_risk

FROM match_context
"""

context_df = pd.read_sql(
    context_query,
    conn
)

matches_features = matches_df.merge(
    context_df,
    left_on="id",
    right_on="match_id",
    how="left"
)

matches_features.columns.tolist()

['id',
 'api_fixture_id',
 'competition_id',
 'season',
 'date',
 'round',
 'home_team_id',
 'away_team_id',
 'home_goals',
 'away_goals',
 'status',
 'referee',
 'match_id',
 'home_position',
 'away_position',
 'points_diff',
 'position_diff',
 'home_points',
 'away_points',
 'home_rest_days',
 'away_rest_days',
 'home_title_race',
 'away_title_race',
 'away_europe_race',
 'home_europe_race',
 'home_relegation_risk',
 'away_relegation_risk']

In [108]:
matches_features["round_str"] = (
    matches_features["round"]
    .astype(str)
)

matches_features["is_playoff"] = (
    matches_features["round_str"]
    .str.contains(
        r"Quarter|Semi|Final|Reclas|Liguilla|Play-In",
        case=False,
        na=False
    )
    .astype(int)
)

matches_features["elo_diff"] = (
    matches_features["home_elo"] -
    matches_features["away_elo"]
).abs()

matches_features["attack_diff"] = (
    matches_features["home_rolling_scored"] -
    matches_features["away_rolling_scored"]
).abs()

matches_features["defense_diff"] = (
    matches_features["home_rolling_conceded"] -
    matches_features["away_rolling_conceded"]
).abs()

matches_features["total_balance"] = (
    matches_features["attack_diff"] +
    matches_features["defense_diff"]
)

matches_features["combined_scoring"] = (
    matches_features["home_rolling_scored"] +
    matches_features["away_rolling_scored"]
)

matches_features["combined_conceded"] = (
    matches_features["home_rolling_conceded"] +
    matches_features["away_rolling_conceded"]
)

matches_features["points_diff_abs"] = (
    matches_features["points_diff"]
).abs()

matches_features["position_diff_abs"] = (
    matches_features["position_diff"]
).abs()

matches_features["draw_rate_diff"] = (
    matches_features["home_draw_rate_last_10"] -
    matches_features["away_draw_rate_last_10"]
).abs()

matches_features["draw_rate_sum"] = (
    matches_features["home_draw_rate_last_10"] +
    matches_features["away_draw_rate_last_10"]
)

matches_features["home_advantage_strength"] = (
    matches_features["home_rolling_scored"] -
    matches_features["away_rolling_conceded"]
)

matches_features["away_advantage_strength"] = (
    matches_features["away_rolling_scored"] -
    matches_features["home_rolling_conceded"]
)

matches_features["strength_gap"] = (
    matches_features["home_advantage_strength"] -
    matches_features["away_advantage_strength"]
).abs()

KeyError: 'home_elo'

In [109]:
matches_features[
    [
        "home_position",
        "away_position",
        "home_rest_days",
        "away_rest_days",
        "home_draw_rate_last_10",
        "away_draw_rate_last_10"
    ]
].isna().sum()

KeyError: "['home_draw_rate_last_10', 'away_draw_rate_last_10'] not in index"